# MaskGCT 声音克隆 - B站开源模型

用Google Colab免费GPU跑B站开源的声音克隆模型 MaskGCT（Amphion框架，9926⭐）

## 操作步骤
1. 依次运行每个代码块（点 ▶ 按钮）
2. 上传马莉老师的声音样本
3. 输入文案
4. 下载生成的克隆语音

In [ ]:
# 1. 克隆Amphion仓库
!git clone https://github.com/open-mmlab/Amphion.git
%cd Amphion
!pip install -q torch torchaudio librosa soundfile
!sh env.sh
print('✅ 环境就绪')

In [ ]:
# 2. 下载MaskGCT模型权重
from huggingface_hub import hf_hub_download

# 下载语义编码器
semantic_ckpt = hf_hub_download('amphion/MaskGCT', filename='semantic_codec/model.safetensors')

# 下载声学编码器
acoustic_ckpt = hf_hub_download('amphion/MaskGCT', filename='acoustic_codec/model.safetensors')

# 下载主模型
model_ckpt = hf_hub_download('amphion/MaskGCT', filename='maskgct/model.safetensors')
print('✅ 模型下载完成')

In [ ]:
# 3. 上传声音样本
from google.colab import files
print('请上传马莉老师的WAV声音样本（5-30秒，清晰人声）')
uploaded = files.upload()
sample_path = list(uploaded.keys())[0]
print(f'✅ 已上传: {sample_path}')

In [ ]:
# 4. 构建MaskGCT模型并推理
import torch
import sys
sys.path.append('/content/Amphion')

from models.tts.maskgct.maskgct_utils import *

# 创建模型
cfg_path = '/content/Amphion/models/tts/maskgct/config/maskgct.json'
cfg = load_config(cfg_path)
model = MaskGCT(cfg)

# 加载权重
import safetensors
device = torch.device('cuda')
model.load_state_dict(safetensors.torch.load_file(model_ckpt), strict=False)
model = model.to(device)
model.eval()

print('✅ 模型加载完成')

In [ ]:
# 5. 生成克隆语音
text = '大家好，我是马莉老师。今天聊聊AI怎么帮你赚钱。核心就三点。第一，找对方向。第二，用好工具。第三，坚持做。'

# 克隆声音+合成
audio = model.clone_voice(
    reference_audio=sample_path,
    text=text,
    language='zh'
)

import soundfile as sf
sf.write('/content/output.wav', audio, 24000)
from IPython.display import Audio, display
display(Audio('/content/output.wav'))
print('✅ 克隆语音已生成!')

In [ ]:
# 6. 下载克隆语音
from google.colab import files
files.download('/content/output.wav')